In [2]:
import os
import scanpy as sc
import pandas as pd

sc_h5ad_path = (
    "/home/qyyuan/project/ST_CP/JE/simulation/MERFISH/"
    "adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad"
)

label_output_path = (
    "/home/qyyuan/project/ST_CP/JE/NC_review/Methods/"
    "Benchmark/SimMerfish/decov/sc_celltype.tsv"
)

if not os.path.isfile(sc_h5ad_path):
    raise FileNotFoundError(sc_h5ad_path)

adata_ref = sc.read_h5ad(sc_h5ad_path)

print("AnnData shape:", adata_ref.shape)
print("obs columns:", adata_ref.obs.columns.tolist())
print(adata_ref.obs.head())

AnnData shape: (4198, 254)
obs columns: ['x_coord', 'y_coord', 'slice_id', 'celltype']
                                             x_coord      y_coord  \
cell_id                                                             
100047181052654346224529175853339022386  4280.917239 -3055.375620   
100144987682981782100923292757524648699  3832.852892 -3637.815477   
100267747580756240041023704492842942950  5048.352454 -2615.303647   
100380418078413692631351412383892166094  4319.829363 -3040.423460   
100416591302420540942373170232872783233  4214.585467 -3425.972085   

                                               slice_id celltype  
cell_id                                                           
100047181052654346224529175853339022386  mouse1_slice50    Oligo  
100144987682981782100923292757524648699  mouse1_slice50     VLMC  
100267747580756240041023704492842942950  mouse1_slice50    Astro  
100380418078413692631351412383892166094  mouse1_slice50  L4/5 IT  
100416591302420540942373170

/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [1]:
candidate_columns = [
    "CellType",
    "celltype",
    "cell_type",
    "celltypes",
    "annotation",
    "annotation_1",
    "label",
    "cluster",
]

matched_columns = [
    c for c in candidate_columns
    if c in adata_ref.obs.columns
]

if not matched_columns:
    raise KeyError(
        "没有自动找到细胞类型列。\n"
        f"当前 obs 列：{adata_ref.obs.columns.tolist()}\n"
        "请根据实际情况手动指定 label_column。"
    )

label_column = matched_columns[0]

print("使用细胞类型列:", label_column)

cell_labels = pd.DataFrame({
    "cell_id": adata_ref.obs_names.astype(str),
    "CellType": adata_ref.obs[label_column].astype(str).to_numpy(),
})

print(cell_labels.head())
print(cell_labels["CellType"].value_counts(dropna=False))

if cell_labels["cell_id"].duplicated().any():
    raise ValueError("h5ad 中存在重复 cell ID")

if cell_labels["CellType"].isin(["nan", "None", ""]).any():
    print("警告：部分细胞没有有效的 CellType。")

os.makedirs(
    os.path.dirname(label_output_path),
    exist_ok=True,
)

cell_labels.to_csv(
    label_output_path,
    sep="\t",
    index=False,
)

print("标签文件已保存:", label_output_path)
print("文件存在:", os.path.isfile(label_output_path))

NameError: name 'adata_ref' is not defined